# Feature Engineering & Fusion Pipeline

**QT-2.21 Quantum-Enhanced Medical Imaging for Cancer Detection**

---

## Introduction

This notebook walks through the **feature engineering pipeline** that powers our hybrid quantum-classical cancer detection system. The core idea is to combine the best of two worlds:

1. **Deep CNN Features** — We leverage a pre-trained **ResNet50** network (trained on ImageNet) and extract high-level semantic features from its penultimate layer. These 2048-dimensional vectors capture complex visual patterns such as tissue architecture, cell morphology, and spatial relationships that are difficult to hand-design.

2. **Handcrafted Texture & Shape Features** — We compute classical image descriptors including **GLCM** (Gray-Level Co-occurrence Matrix) texture statistics, **HOG** (Histogram of Oriented Gradients) shape descriptors, **intensity statistics**, and **Discrete Wavelet Transform (DWT)** coefficients. These 98-dimensional feature vectors encode domain-specific information about texture regularity, edge distributions, and multi-scale frequency content.

The two feature sets are then **fused** by concatenation into a single **2146-dimensional** feature vector per image, followed by **StandardScaler normalization**. This fused representation feeds into our downstream classical feature selection and quantum optimization stages.

### Pipeline Overview

```
Medical Image
     │
     ├──► ResNet50 (pretrained) ──► 2048-dim deep features
     │
     └──► Handcrafted Extractors ──► 98-dim texture/shape features
              ├─ GLCM (20 features)
              ├─ HOG (36 features)
              ├─ Intensity Stats (10 features)
              └─ DWT Wavelets (32 features)
                        │
                        ▼
              Concatenation ──► 2146-dim fused vector
                        │
                        ▼
              StandardScaler normalization
                        │
                        ▼
              Ready for feature selection & quantum optimization
```

---

## 1. Setup & Imports

We begin by importing our project modules alongside standard scientific computing libraries.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing.dataset import get_dataloaders
from src.features.cnn_features import CNNFeatureExtractor
from src.features.handcrafted import extract_all_handcrafted, extract_glcm_features, extract_hog_features, extract_intensity_features, extract_wavelet_features
from src.features.feature_combiner import extract_and_combine_features

---

## 2. Load Dataset

We use our custom `get_dataloaders` utility to load the pre-processed medical imaging dataset. This returns PyTorch `DataLoader` objects for the train, validation, and test splits, along with the class-to-index mapping.

In [ ]:
train_loader, val_loader, test_loader, class_mapping = get_dataloaders('../data/raw', batch_size=8)
print(f'Class mapping: {class_mapping}')
print(f'Training batches: {len(train_loader)}')

---

## 3. Deep CNN Features

### Why ResNet50?

Transfer learning with deep CNNs has become a cornerstone of medical image analysis. **ResNet50**, pre-trained on ImageNet's 1.2 million images across 1000 categories, has learned a rich hierarchy of visual features:

| Layer Depth | Feature Type | Examples |
|-------------|-------------|----------|
| Early layers | Low-level | Edges, corners, color gradients |
| Middle layers | Mid-level | Textures, patterns, shapes |
| Deep layers | High-level | Object parts, semantic concepts |
| Penultimate layer | Global | Holistic image representation |

### Extraction Strategy

We remove the final classification head of ResNet50 and use the output of the **global average pooling layer** (the penultimate layer) as our feature representation. This yields a **2048-dimensional** vector for each input image that encodes high-level semantic content.

Key considerations:
- **No fine-tuning**: We use the frozen pre-trained weights, treating the network purely as a feature extractor. This avoids overfitting on small medical datasets.
- **Channel adaptation**: If input images are grayscale (1-channel), we replicate across 3 channels to match ResNet50's expected input.
- **GPU acceleration**: Feature extraction is performed on GPU when available for efficient batch processing.

In [ ]:
import torch
extractor = CNNFeatureExtractor()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
extractor.model.to(device)

# Extract features from first batch
images, labels = next(iter(train_loader))
images_rgb = images.repeat(1, 3, 1, 1) if images.shape[1] == 1 else images
with torch.no_grad():
    features = extractor.model(images_rgb.to(device))
print(f'CNN feature shape per image: {features.shape}')
print(f'Feature dimensionality: {features.shape[1]}')

---

## 4. Handcrafted Texture & Shape Features

While deep features capture high-level semantics, handcrafted descriptors provide **interpretable, domain-specific** information that complements the CNN representation. We compute four families of features:

### 4.1 GLCM — Gray-Level Co-occurrence Matrix (20 features)

GLCM captures **spatial texture relationships** by computing how often pairs of pixel intensities occur at specified distances and orientations. From each GLCM, we extract statistical properties:

- **Contrast** — measures intensity differences between neighboring pixels
- **Dissimilarity** — similar to contrast but with linear weighting
- **Homogeneity** — measures closeness of element distribution to the diagonal
- **Energy** — measures textural uniformity (sum of squared elements)
- **Correlation** — measures linear dependency of gray levels

We compute these across multiple distances and angles, yielding **20 features** total.

### 4.2 HOG — Histogram of Oriented Gradients (36 features)

HOG captures **local shape and edge distributions** by:
1. Computing gradient magnitudes and orientations across the image
2. Dividing the image into spatial cells
3. Building orientation histograms within each cell
4. Normalizing across blocks for illumination invariance

This is particularly useful for detecting structural patterns in tissue architecture.

### 4.3 Intensity Statistics (10 features)

Basic but powerful statistical descriptors of the pixel intensity distribution:
- Mean, standard deviation, skewness, kurtosis
- Percentiles (5th, 25th, 50th, 75th, 95th)
- Entropy (information content)

### 4.4 DWT — Discrete Wavelet Transform (32 features)

Wavelet decomposition provides **multi-scale frequency analysis**:
- Decomposes the image into approximation and detail coefficients at multiple levels
- Captures both low-frequency structure and high-frequency texture details
- Statistical summaries of each sub-band yield 32 features

In [ ]:
# Get a single image as numpy
sample_img = images[0].squeeze().numpy()

# GLCM
glcm_feats = extract_glcm_features(sample_img)
print(f'GLCM features: {glcm_feats.shape} -> {glcm_feats}')

# HOG
hog_feats = extract_hog_features(sample_img)
print(f'HOG features: {hog_feats.shape}')

# Intensity
intensity_feats = extract_intensity_features(sample_img)
print(f'Intensity features: {intensity_feats.shape} -> {intensity_feats}')

# Wavelet
wavelet_feats = extract_wavelet_features(sample_img)
print(f'Wavelet features: {wavelet_feats.shape}')

# All handcrafted
all_hc = extract_all_handcrafted(sample_img)
print(f'\nTotal handcrafted features: {all_hc.shape[0]} (GLCM: 20 + HOG: 36 + Intensity: 10 + Wavelet: 32 = 98)')

---

## 5. Feature Fusion

### Fusion Strategy

We adopt an **early fusion** strategy — concatenating the deep CNN features and handcrafted features into a single vector before any downstream processing:

$$\mathbf{x}_{\text{fused}} = [\mathbf{x}_{\text{CNN}} \, \| \, \mathbf{x}_{\text{handcrafted}}] \in \mathbb{R}^{2146}$$

where:
- $\mathbf{x}_{\text{CNN}} \in \mathbb{R}^{2048}$ — ResNet50 penultimate layer output
- $\mathbf{x}_{\text{handcrafted}} \in \mathbb{R}^{98}$ — concatenated GLCM + HOG + Intensity + Wavelet features
- $\|$ denotes concatenation

### Normalization

Since CNN features and handcrafted features live on very different scales, we apply **StandardScaler** normalization (zero mean, unit variance) to the fused feature matrix. This is critical because:

1. **Prevents scale dominance** — Without normalization, the 2048 CNN features (typically in the range [0, ~10]) could dominate the smaller set of handcrafted features
2. **Improves optimization** — Both classical and quantum classifiers converge faster with standardized inputs
3. **Fitted on training data only** — The scaler is fitted on the training set and applied to the test set to prevent data leakage

The `extract_and_combine_features` function orchestrates this entire pipeline, including saving the scaler for inference time.

In [ ]:
X_train, y_train, X_test, y_test = extract_and_combine_features(
    train_loader, test_loader, save_dir='../models'
)
print(f'Fused training features: {X_train.shape}')
print(f'Fused test features: {X_test.shape}')
print(f'Feature dimensionality: {X_train.shape[1]}')

---

## 6. Visualize Feature Distributions

Let's inspect the distributions of deep CNN features vs. handcrafted features after fusion and normalization. This helps us verify that both feature sets contribute meaningful signal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of deep features (first 2048)
axes[0].hist(X_train[0, :2048], bins=50, alpha=0.7, color='steelblue')
axes[0].set_title('Deep CNN Feature Distribution (Sample 0)')
axes[0].set_xlabel('Feature Value')
axes[0].set_ylabel('Count')

# Distribution of handcrafted features (last 98)
axes[1].hist(X_train[0, 2048:], bins=30, alpha=0.7, color='coral')
axes[1].set_title('Handcrafted Feature Distribution (Sample 0)')
axes[1].set_xlabel('Feature Value')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---

## Summary

In this notebook, we have built and demonstrated the complete **feature engineering pipeline** for the QT-2.21 Quantum-Enhanced Medical Imaging system:

| Stage | Method | Dimensions | Purpose |
|-------|--------|------------|--------|
| Deep features | ResNet50 (pretrained, frozen) | 2048 | High-level semantic visual patterns |
| GLCM | Gray-Level Co-occurrence Matrix | 20 | Spatial texture relationships |
| HOG | Histogram of Oriented Gradients | 36 | Local shape & edge structure |
| Intensity | Statistical descriptors | 10 | Pixel distribution characteristics |
| Wavelet | Discrete Wavelet Transform | 32 | Multi-scale frequency content |
| **Fused** | **Concatenation + StandardScaler** | **2146** | **Unified representation** |

### Key Takeaways

1. **Hybrid feature design** — Combining deep learned features with handcrafted descriptors provides a richer representation than either alone, capturing both high-level semantics and low-level texture/shape information.

2. **Normalization is critical** — StandardScaler ensures that no feature subset dominates due to scale differences, which is especially important for quantum circuit encodings.

3. **Ready for downstream processing** — The resulting **2146-dimensional fused feature matrix** is now ready for:
   - **Classical feature selection** (e.g., mutual information, variance thresholding) to reduce dimensionality
   - **Quantum optimization** via variational quantum circuits for classification
   - **Hybrid quantum-classical models** that leverage quantum advantage for the most informative feature subsets

### Next Steps

Proceed to **`04_quantum_classification.ipynb`** to see how these fused features are encoded into quantum circuits and optimized for cancer detection.